# Aggregate 질문: 울타리에 몇 번 올라탔나요?

행동 횟수는 검색 상위 프레임 수나 `repeat_count`가 아니라, 질문에 맞는 행동 이벤트를 전체 기간에서 찾아 겹치는 구간을 정리해 계산합니다. 이 노트북은 `dog_escape` 영상의 울타리 올라타기를 발표 사례로 사용합니다.


In [1]:
import sys
from pathlib import Path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))
from pipeline.search.aggregate_query import parse_count_action, event_matches_action, count_episodes

question = '강아지가 울타리에 몇 번 올라탔어?'
events = [
    {'id': 1, 'video_id': 'dog_escape', 'start_time': 6, 'end_time': 11, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 8},
    {'id': 2, 'video_id': 'dog_escape', 'start_time': 8, 'end_time': 13, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 3},
    {'id': 3, 'video_id': 'dog_escape', 'start_time': 25, 'end_time': 31, 'action': '강아지가 울타리에 올라탐', 'repeat_count': 5},
    {'id': 4, 'video_id': 'dog_escape', 'start_time': 32, 'end_time': 34, 'action': '강아지가 울타리 근처에 있음', 'repeat_count': 1},
]
action, label = parse_count_action(question)
matched = [event for event in events if event_matches_action(event, action)]
episodes = count_episodes(matched)
print('질문:', question)
print('행동 후보 ID:', [event['id'] for event in matched])
print('정리된 구간:', [(e['start_time'], e['end_time'], e['event_ids']) for e in episodes])
print('확인된 후보 구간 수:', len(episodes))

질문: 강아지가 울타리에 몇 번 올라탔어?
행동 후보 ID: [1, 2, 3]
정리된 구간: [(6.0, 13.0, [1, 2]), (25.0, 31.0, [3])]
확인된 후보 구간 수: 2


위의 **가상 이벤트**에서는 1·2번 기록이 겹쳐 한 구간이고 3번은 별도 구간입니다. 4번은 울타리 근처에만 있어 제외합니다. 따라서 후보 구간은 2개입니다. 이 숫자는 실제 영상의 검증된 정답이 아닙니다.

## 실제 DB 이벤트 조회

DB가 연결된 실행 환경에서 아래 `demo_video_id`를 설정해 실제 이벤트 후보를 조회합니다. 지금 로컬 Python에는 DB 드라이버가 설치되어 있지 않아 예제 셀을 기본적으로 건너뜁니다.


In [2]:
demo_video_id = None  # DB가 연결되면 'dog_escape'로 변경
if demo_video_id:
    from db.behavior_events import get_behavior_events_for_aggregation
    timeline = get_behavior_events_for_aggregation(video_id=demo_video_id)
    matched = [event for event in timeline if event_matches_action(event, 'gate_climb')]
    episodes = count_episodes(matched)
    for episode in episodes:
        print(episode['video_id'], episode['start_time'], episode['end_time'], episode['event_ids'])
    print('영상에서 대조할 후보 구간:', len(episodes))
else:
    print('DB 연결을 설정하면 실제 이벤트를 조회할 수 있습니다.')


DB 연결을 설정하면 실제 이벤트를 조회할 수 있습니다.


## 원본 영상 검토: dog_escape

이 영상의 길이는 약 56초입니다. 처음 약 36초에 울타리 장면이 나오고, 이후에는 도어벨 화면 등 다른 장면으로 바뀝니다. 아래 그림은 0~36초를 1초 간격으로 뽑은 샘플입니다. 샘플만 보고 잡은 구간은 약 6~13초와 25~31초였지만, **영상 재생으로 검토한 구간은 4~16초와 22~31초**입니다. 두 구간은 떨어져 있으므로 검토 결과는 **2회**입니다.

“울타리에 올라탐”은 앞발이나 몸을 울타리 위에 올린 때부터 떨어질 때까지로 정의합니다. 단순히 근처에 있는 장면은 제외합니다. 성공적으로 울타리를 넘었는지는 이 질문에서 세지 않습니다. 이 검토는 사용자가 원본 영상을 재생하며 확인한 잠정 라벨이며, 별도 검토자의 재확인은 아직 없습니다.

![dog_escape 첫 36초의 1초 간격 샘플](assets/dog_escape_gate_review.jpg)


In [ ]:
from IPython.display import Video, display
video_files = sorted((Path.home() / 'Downloads').glob('chunks_dog_escape_*.mp4'))
print('찾은 원본 영상:', [path.name for path in video_files])
if video_files:
    display(Video(filename=str(video_files[0]), embed=True, width=640))
else:
    print('원본 영상이 없습니다. 로컬 영상 경로를 지정하세요.')


In [3]:
# 원본 영상 재생 후 사용자가 확인한 잠정 행동 구간 (영상 시작 기준 초)
verified_intervals = [
    {'video_id': 'dog_escape', 'start_time': 4.0, 'end_time': 16.0, 'action': 'gate_climb', 'note': '울타리에 올라탄 뒤 떨어짐'},
    {'video_id': 'dog_escape', 'start_time': 22.0, 'end_time': 31.0, 'action': 'gate_climb', 'note': '다시 올라탄 뒤 떨어짐'},
]
verified_episodes = count_episodes(verified_intervals)
print('영상 검토 구간:', [(e['start_time'], e['end_time']) for e in verified_episodes])
print('영상 검토 횟수:', len(verified_episodes))


영상 검토 구간: [(4.0, 16.0), (22.0, 31.0)]
영상 검토 횟수: 2


## 실제 DB와 비교한 결과

2026-11-03에 `dog_escape`의 `behavior_events`를 읽으니 **이벤트가 1개**였습니다. 이벤트 ID 5는 **2.0~56.06초** 전체를 덮고, 행동은 “반려동물이 장애물을 넘어서는 것으로 보임”으로 저장되어 있습니다. 이 한 기록에는 사용자가 영상에서 확인한 4~16초와 22~31초를 따로 구분할 정보가 없습니다. 현재 API는 이 질문에 대해 숫자를 만들지 않고 **횟수 계산 불가**라고 답합니다. 따라서 현재 결과는 정답 2회와 일치하지 않습니다. 원인은 횟수 합산보다 앞 단계인 **행동 구간 분할과 행동 이름 지정**에 있습니다.


In [4]:
# 실제 DB 조회 결과의 재현용 스냅샷. DB를 수정하거나 재조회하지 않습니다.
db_snapshot = [{'id': 5, 'video_id': 'dog_escape', 'start_time': 2.0, 'end_time': 56.06, 'action': '반려동물이 장애물을 넘어서는 것으로 보임', 'summary': '반려동물이 장애물을 넘으려는 시도를 하며 활동적인 모습을 보임.', 'target_object': '반려동물의 장난감이나 식기'}]
db_matches = [event for event in db_snapshot if event_matches_action(event, 'gate_climb')]
print('DB 이벤트 수:', len(db_snapshot))
print('울타리 올라타기로 구분된 DB 이벤트 수:', len(db_matches))
print('현재 API 판단: 횟수 계산 불가')
print('원본 영상 검토 횟수:', len(verified_episodes))


DB 이벤트 수: 1
울타리 올라타기로 구분된 DB 이벤트 수: 0
현재 API 판단: 횟수 계산 불가
원본 영상 검토 횟수: 2


## 긴 이벤트를 짧은 창으로 나누기

기존 추출기는 2.0~56.06초를 이벤트 하나로 요약했습니다. 수정한 추출기는 긴 움직임 구간을 최대 12초 창으로 나누고 2초씩 겹치게 합니다. 실제 `dog_escape` 앞 36초에서 2초 간격으로 추출한 18개 프레임은 아래 네 창으로 나뉩니다.

| 창 | 분석 범위 | 프레임 수 |
|---|---:|---:|
| 1 | 0~12초 | 7 |
| 2 | 10~22초 | 7 |
| 3 | 20~32초 | 7 |
| 4 | 30~36초 | 3 |

로컬 모의 추출에서는 사용자가 확인한 두 구간을 겹친 창에 넣어도 다시 2개로 합쳐졌습니다. 아래에는 사용자 승인 후 같은 프레임을 VLM에 보낸 **실제 시범 결과**를 별도로 기록합니다.


In [5]:
from pipeline.events.behavior_event_extractor import _split_long_group
sampled_frames = [{'video_id': 'dog_escape', 'timestamp': float(t), 'motion_start': 0.0, 'motion_end': 36.0} for t in range(0, 36, 2)]
windows = _split_long_group(sampled_frames)
print('분석 창:', [(w[0]['motion_start'], w[0]['motion_end'], len(w)) for w in windows])
print('영상 검토 기준 횟수:', len(verified_episodes))


분석 창: [(0.0, 12.0, 7), (10.0, 22.0, 7), (20.0, 32.0, 7), (30.0, 36.0, 3)]
영상 검토 기준 횟수: 2


## VLM 시범 결과와 영상 검토 비교

사용자 승인 후 원본 영상 앞 36초에서 2초 간격으로 뽑은 프레임을 `gpt-4o-mini`에 보내 시범 분석했습니다. 모델은 원시 이벤트 5개를 반환했습니다. 울타리 관련 표현을 정규화하고 겹친 시간을 합치면 후보 구간은 **10~20초, 22~32초의 2개**입니다. DB에는 저장하지 않았습니다.

| 기준 | 첫 번째 | 두 번째 | 횟수 |
|---|---:|---:|---:|
| 영상 검토 | 4~16초 | 22~31초 | 2회 |
| VLM 후보 | 10~20초 | 22~32초 | 2회 |

횟수는 맞지만 첫 행동의 시작은 6초 늦고 종료는 4초 늦습니다. 이 시범 실행 하나만으로 다른 영상의 정확도를 보장할 수는 없습니다. 기존 DB에는 여전히 2.0~56.06초 이벤트 하나만 있으므로 현재 서비스 응답은 자동으로 바뀌지 않습니다.


In [6]:
import json
preview = json.loads((project_root / 'notebooks/aggregate-gate-vlm-preview.json').read_text())
preview_matched = [event for event in preview['events'] if event_matches_action(event, 'gate_climb')]
preview_episodes = count_episodes(preview_matched)
print('VLM 원시 이벤트 수:', len(preview['events']))
print('VLM 후보 구간:', [(e['start_time'], e['end_time']) for e in preview_episodes])
print('영상 검토 구간:', [(e['start_time'], e['end_time']) for e in verified_episodes])
print('횟수 일치:', len(preview_episodes) == len(verified_episodes))


VLM 원시 이벤트 수: 5
VLM 후보 구간: [(10.0, 20.0), (22.0, 32.0)]
영상 검토 구간: [(4.0, 16.0), (22.0, 31.0)]
횟수 일치: True


## 발표 결론

기존 DB의 긴 이벤트 1개로는 횟수를 셀 수 없었습니다. 짧은 창으로 다시 분석한 시범 실행은 후보 2개를 찾아 영상 검토 횟수 2회와 일치했습니다. 다만 첫 구간의 시작·끝 시각은 정확하지 않습니다. 아래 수치는 이번 영상 **한 건**의 결과이며, 기존 DB와 배포된 서비스는 아직 갱신되지 않았습니다.


In [7]:
def interval_iou(left, right):
    overlap = max(0.0, min(left['end_time'], right['end_time']) - max(left['start_time'], right['start_time']))
    union = max(left['end_time'], right['end_time']) - min(left['start_time'], right['start_time'])
    return overlap / union if union else 0.0

pairs = list(zip(preview_episodes, verified_episodes))
print('영상 검토 횟수:', len(verified_episodes))
print('VLM 후보 횟수:', len(preview_episodes))
print('횟수 오차:', abs(len(preview_episodes) - len(verified_episodes)))
print('구간별 시간 겹침(IoU):', [round(interval_iou(candidate, reviewed), 3) for candidate, reviewed in pairs])
print('평균 시간 겹침(IoU):', round(sum(interval_iou(candidate, reviewed) for candidate, reviewed in pairs) / len(pairs), 3))


영상 검토 횟수: 2
VLM 후보 횟수: 2
횟수 오차: 0
구간별 시간 겹침(IoU): [0.375, 0.9]
평균 시간 겹침(IoU): 0.637
